# 32_operating_point_CHS — 운영점 설계: 정상 기준 목표 오경보율로 정하는 주의·경보 임계

- 목적: 임계를 "train 정상 분위수 q"가 아니라 **현장이 감당할 오경보율**(세그먼트 단위, 정상 데이터만으로 정의)로 정하는 운영점 절차를 만들고, 그 운영점에서 탐지율·지연·시간당 오경보 수를 보고한다. 이상 데이터는 임계 선택에 쓰지 않는다(`evaluate.threshold_for_segment_fpr`).
- 입력: `data/processed/11_window_features.parquet`(1초 윈도우), `data/processed/segments.csv`, `results/31_error_analysis_CHS/f1_sensitivity_grid.csv`·`f1_matched_fpr.csv`(참조·대조용)
- 출력: `figures/32_operating_point_CHS/`, `results/32_operating_point_CHS/`(CSV 5개)
- 담당: CHS
- 심사 대응: 4번(경보 임계·조치 프로세스) 중심, 5번(운영점 관점의 새로운 접근), 3번 보조(운전 상태별 오경보 분포)

### 설계 전제

- **JIW 28 앙상블(`reports/28_model_ensemble_JIW.md`)과의 경계**: 28은 *어떤 모델을 어떻게 결합*(CNN OR/AND MCD, 상태별 q99, k-of-n; 2초 윈도우)할지를 다룬다. 32는 *어떤 점수든 임계를 어떻게 정할지*(목표 오경보율 운영점, 같은 점수의 주의·경보 2임계, 시간당 오경보 환산)를 다룬다. 두 설계는 합성 가능하다: 28은 각 모델의 q99 임계를 고정한 뒤 경보를 논리 결합(AND-고정)하고 결합 후 임계 재설정은 기각했으므로, 합성 위치는 **28의 개별 모델 임계를 q99 대신 목표 오경보율로 정하는 것**이다. 28의 확정 2단계(주의 = CNN 단독, 경보 = CNN AND MCD; OR 단계는 H5에서 폐기)는 모델 결합 등급이고 32의 주의·경보는 같은 점수의 임계 2개라 정의가 다르며, B절 상태별 임계는 28 H4(상태별 q99)의 목표 오경보율 판이다. 집계 규칙(kofn·skip_first)은 31 A-4에서 이득이 없었으므로 **any 고정**.
- 대상 모델: train 정상 점수를 노트북 안에서 만들 수 있는 `rule1_rms_ai0`(`AI0_Vibration_rms`)와 `pca_spe_amp18`(31 E절과 같은 amp18 StandardScaler+PCA 95% SPE). 24~27은 train 정상 점수가 없어 제외한다.
- 분할·분모: 31과 동일. gkf 5 fold(train = 다른 fold 정상, test = 해당 fold 전체, fold별 경보 표를 합쳐 `segment_confusion(n_outlier_total=21)` 1회)와 time_block k=1..4(train = block<k 정상, test = block==k 정상 + 이상 전체, 블록별 confusion 후 블록 평균). 1초 윈도우만.
- 시간당 오경보: 정상 세그먼트가 차지한 **벽시계 구간**과 판정 가능(1초 윈도우 있음) 정상 세그먼트 수(530, `fpr_segment`의 분모와 같은 집합)로 `SEG_PER_HOUR`를 계산하고 `fp_per_hour = fpr_segment × SEG_PER_HOUR`로 환산한다(정상 수집 구간 기준 외삽). 체감 지연 = 윈도우 내 지연 + `evaluate.BURST_GAP_S`.
- 운전 상태 추정: 추론 시 쓸 수 있는 값만 사용 — 윈도우 `AI2_Current_rms > 125.2054` → 고부하(0), 아니면 저부하(1)(21 V2-b 2-means 경계). 상태 번호는 parquet 규약(0 = 고부하, 1 = 저부하).
- **결과를 보고 목표값·규칙을 바꾸지 않는다**: 목표 오경보율 격자, 주의 = 0.04 · 경보 = 0.01은 사전 고정.

In [1]:
import sys, warnings, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
warnings.filterwarnings("ignore")
T0 = time.time()
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import paths
NB = "32_operating_point_CHS"  # 파일명(확장자 제외)과 같게
FIG, RES = paths.nb_dirs(NB)
SEED = 42
np.random.seed(SEED)
_fonts = {f.name for f in fm.fontManager.ttflist}
plt.rcParams["font.family"] = next((f for f in ("Malgun Gothic", "AppleGothic") if f in _fonts), "DejaVu Sans")
plt.rcParams["axes.unicode_minus"] = False
import evaluate, features, split, stats
PQ = paths.DATA_PROCESSED / "11_window_features.parquet"
assert PQ.exists(), "11_window_features.parquet 필요 (src/extract.py -> 11 노트북 순차 실행)"
REF31 = paths.RESULTS / "31_error_analysis_CHS"
print(NB, FIG.relative_to(ROOT), RES.relative_to(ROOT), "| font:", plt.rcParams["font.family"])

32_operating_point_CHS figures\32_operating_point_CHS results\32_operating_point_CHS | font: ['Malgun Gothic']


### 0. 데이터 로드 · 시간당 환산 계수 · 운전 상태 추정

In [2]:
W_ALL = pd.read_parquet(PQ)
W1 = W_ALL[W_ALL.win_s == 1.0].reset_index(drop=True)
S = pd.read_csv(paths.DATA_PROCESSED / "segments.csv", encoding="utf-8-sig")
FOLD_OF = split.fold_assignment(S, n_splits=5, seed=42)

# 1) 시간당 환산 계수: 정상 세그먼트가 차지한 벽시계 구간 ÷ 판정 가능(1초 윈도우 있음, len >= 10) 정상 세그먼트 수
#    fpr_segment의 분모가 판정 가능 세그먼트이므로 분자(세그먼트/시간)도 같은 집합으로 센다
SN = S[S.src == "normal"].copy()
SN["t0"] = pd.to_datetime(SN.start)
SN["t1"] = SN.t0 + pd.to_timedelta(SN["len"] / 10.0, unit="s")
N_NORMAL_SEG = len(SN)
N_ASSESSABLE = int((SN["len"] >= 10).sum())                      # 1초 윈도우가 있는 정상 세그먼트(= fpr_segment 분모 530)
WALL_MIN = (SN.t1.max() - SN.t0.min()).total_seconds() / 60.0
SEG_PER_HOUR = N_ASSESSABLE / (WALL_MIN / 60.0)
print(f"정상 세그먼트 {N_NORMAL_SEG}개(판정 가능 {N_ASSESSABLE}개) | 벽시계 구간 {SN.t0.min()} ~ {SN.t1.max()} = {WALL_MIN:.2f}분 | SEG_PER_HOUR = {SEG_PER_HOUR:.2f} 판정 가능 세그먼트/시간 (전체 기준 {N_NORMAL_SEG / (WALL_MIN / 60.0):.2f})")
assert N_NORMAL_SEG == 599 and N_ASSESSABLE == 530 and 70 <= WALL_MIN <= 85, "정상 수집 구간이 70~85분 범위가 아님"

# 2) 운전 상태 추정 (추론 시 쓸 수 있는 값만): 윈도우 AC RMS 경계
STATE_BOUNDARY = 125.2054
W1["state_hat"] = np.where(W1["AI2_Current_rms"] > STATE_BOUNDARY, 0, 1).astype(int)
print("AI2_Current_rms NaN 윈도우:", int(W1["AI2_Current_rms"].isna().sum()))
nz = W1[W1.label == 0]
agree = float((nz.state_hat == nz.state).mean())
print(f"정상 윈도우 {len(nz)}개: state_hat vs parquet state(세그먼트 k-means) 일치율 = {agree:.4f}")
print(pd.crosstab(nz.state, nz.state_hat, rownames=["parquet state"], colnames=["state_hat"]))
assert agree >= 0.90
# 세그먼트 대표 상태 = 윈도우 state_hat 최빈값 (동률이면 고부하 0)
REP_STATE = W1.groupby("seg_uid").state_hat.agg(lambda x: int(x.mean() > 0.5))
print("세그먼트 대표 상태(정상):", REP_STATE[REP_STATE.index.str.startswith("normal")].value_counts().sort_index().to_dict())

정상 세그먼트 599개(판정 가능 530개) | 벽시계 구간 2022-07-12 00:00:00.019000 ~ 2022-07-12 01:16:55.928000 = 76.93분 | SEG_PER_HOUR = 413.35 판정 가능 세그먼트/시간 (전체 기준 467.17)
AI2_Current_rms NaN 윈도우: 0
정상 윈도우 3264개: state_hat vs parquet state(세그먼트 k-means) 일치율 = 0.9442
state_hat         0     1
parquet state            
0.0            1246   176
1.0               6  1836
세그먼트 대표 상태(정상): {0: 205, 1: 325}


In [3]:
def pca_spe_t2(train: pd.DataFrame, X: pd.DataFrame, cols, var: float = 0.95) -> tuple[pd.DataFrame, int]:
    """31 셀 3과 같은 정의: train 정상으로 StandardScaler+PCA(누적 분산 var 성분 수 k) 적합 -> X의 SPE(Q)·Hotelling T^2."""
    k = stats.pca_fit_report(train, cols, var_targets=(var,))["n_for"][var]
    rep = stats.pca_fit_report(train, cols, n_components=k)
    x = X[cols].to_numpy(dtype=float)
    ok = ~np.isnan(x).any(axis=1)
    spe = len(cols) * stats.pca_recon_error(rep, x) ** 2
    t2 = np.full(len(x), np.nan)
    z = rep["scaler"].transform(x[ok])
    t2[ok] = ((rep["pca"].transform(z) ** 2) / rep["pca"].explained_variance_).sum(axis=1)
    return pd.DataFrame({"spe": spe, "t2": t2}, index=X.index), k

FEATS = features.feature_columns(W_ALL)
AMP18 = [c for c in FEATS if c.startswith(("AI0_", "AI1_", "AI2_")) and "_shape_" not in c and c.rsplit("_", 1)[-1] in features.AMP_KEYS]
assert len(AMP18) == 18

def pca_spe_scores(trn, te):
    """train 정상으로 적합한 amp18 PCA SPE를 train·test 한 번에 점수화(31 A-3 score_fold의 pca 부분)."""
    fill = trn[AMP18].median()
    X = pd.concat([trn[AMP18], te[AMP18]], ignore_index=True).fillna(fill)
    spe = pca_spe_t2(trn[AMP18].fillna(fill), X, AMP18)[0].spe.to_numpy()
    return spe[:len(trn)], spe[len(trn):]

def score_fold(model, trn, te):
    if model == "rule1_rms_ai0":
        return trn["AI0_Vibration_rms"].to_numpy(float), te["AI0_Vibration_rms"].to_numpy(float)
    return pca_spe_scores(trn, te)

def split_iter(sp):
    """(fold, train 정상 윈도우, test 윈도우) — 21·31과 같은 정의."""
    if sp == "group_kfold_seg":
        for k in range(5):
            yield k, W1[(W1.fold != k) & (W1.label == 0)], W1[W1.fold == k]
    else:
        for k in range(1, 5):
            te_idx = split.append_outliers(np.flatnonzero((W1.block == k).fillna(False).to_numpy(bool)), W1)
            yield k, W1[(W1.block < k).fillna(False) & (W1.label == 0)], W1.iloc[te_idx]

MODELS = ("rule1_rms_ai0", "pca_spe_amp18")
SPLITS = ("group_kfold_seg", "time_block")
SCORES = {}                                   # (model, split, fold) -> (trn, s_train, te, s_test)
for m in MODELS:
    for sp in SPLITS:
        for k, trn, te in split_iter(sp):
            s_tr, s_te = score_fold(m, trn, te)
            SCORES[(m, sp, k)] = (trn, s_tr, te, s_te)
print("점수 계산 완료:", len(SCORES), "fold 조합")

def eval_fold(an_kw, thr, te, s_te):
    nrm, otl = te.label.to_numpy() == 0, te.label.to_numpy() == 1
    an = evaluate.segment_alarms(s_te[nrm], te.seg_uid.to_numpy()[nrm], te.t_start.to_numpy()[nrm], thr, win_s=1.0, **an_kw)
    ao = evaluate.segment_alarms(s_te[otl], te.seg_uid.to_numpy()[otl], te.t_start.to_numpy()[otl], thr, win_s=1.0, **an_kw)
    return an, ao

def state_counts(an):
    """정상 test 세그먼트 경보 표 -> 대표 상태별 분모·오경보 세그먼트 수."""
    a = an[an.n_windows > 0]
    r = a.seg_uid.map(REP_STATE)
    return {"fp_h": int((a.alarm & (r == 0)).sum()), "fp_l": int((a.alarm & (r == 1)).sum()),
            "n_h": int((r == 0).sum()), "n_l": int((r == 1).sum())}

def combine(m, sp, fold_fn):
    """fold_fn(trn, s_tr, te, s_te) -> (an, ao, thr). gkf 합산 / time_block 블록 평균 한 행(dict).
    31 A-3·A-4와 같은 정의에 fold별 thr 목록(_thrs)·경보 표(_ans/_aos)·train 정상 세그먼트 수(_ntr)·상태별 FP 수를 덧붙인다."""
    ans, aos, confs, delays, thrs, ntr, sts = [], [], [], [], [], [], []
    for k, trn, te in split_iter(sp):
        _, s_tr, _, s_te = SCORES[(m, sp, k)]
        an, ao, thr = fold_fn(trn, s_tr, te, s_te)
        thrs.append(thr); ntr.append(trn.seg_uid.nunique())
        ans.append(an); aos.append(ao)
        if sp != "group_kfold_seg":
            d = ao.loc[ao.alarm, "delay_in_window_s"]
            confs.append(evaluate.segment_confusion(an, ao, n_outlier_total=21))
            delays.append(float(d.median()) if len(d) else np.nan)
            sts.append(state_counts(an))
    if sp == "group_kfold_seg":
        an, ao = pd.concat(ans, ignore_index=True), pd.concat(aos, ignore_index=True)
        c = evaluate.segment_confusion(an, ao, n_outlier_total=21)
        d = ao.loc[ao.alarm, "delay_in_window_s"]
        c["delay_median_s"] = float(d.median()) if len(d) else np.nan
        c["f1_agg"] = "gkf_pooled"
        sc = state_counts(an)
        c.update({"fp_seg_high": sc["fp_h"], "fp_seg_low": sc["fp_l"],
                  "fpr_seg_high": sc["fp_h"] / sc["n_h"], "fpr_seg_low": sc["fp_l"] / sc["n_l"]})
    else:
        c = {key: float(np.mean([x[key] for x in confs])) for key in confs[0]}
        c["delay_median_s"] = float(np.nanmean(delays)) if not np.all(np.isnan(delays)) else np.nan
        c["f1_agg"] = "block_mean"
        c.update({"fp_seg_high": float(np.mean([s["fp_h"] for s in sts])), "fp_seg_low": float(np.mean([s["fp_l"] for s in sts])),
                  "fpr_seg_high": float(np.mean([s["fp_h"] / s["n_h"] for s in sts])),
                  "fpr_seg_low": float(np.mean([s["fp_l"] / s["n_l"] for s in sts]))})
    c["thr_mean"] = float(np.mean(thrs))
    c.update({"_thrs": thrs, "_ans": ans, "_aos": aos, "_ntr": ntr})
    return c

def thr_global(trn, s_tr, tgt):
    return evaluate.threshold_for_segment_fpr(s_tr, trn.seg_uid.to_numpy(), trn.t_start.to_numpy(), tgt, win_s=1.0)

def train_fpr(trn, s_tr, thr):
    a = evaluate.segment_alarms(s_tr, trn.seg_uid.to_numpy(), trn.t_start.to_numpy(), thr, win_s=1.0)
    return float(a[a.n_windows > 0].alarm.mean())

RUN_G = {}                                    # (model, split, target) -> combine 결과 (global)
def run_global(m, sp, tgt):
    key = (m, sp, tgt)
    if key not in RUN_G:
        def ff(trn, s_tr, te, s_te):
            thr = thr_global(trn, s_tr, tgt)
            an, ao = eval_fold({}, thr, te, s_te)
            return an, ao, thr
        RUN_G[key] = combine(m, sp, ff)
    return RUN_G[key]
print("헬퍼 정의 완료 | SEG_PER_HOUR =", round(SEG_PER_HOUR, 2))

점수 계산 완료:

 18 fold 조합
헬퍼 정의 완료 | SEG_PER_HOUR = 413.35


### A. 운영 곡선 — 목표 오경보율별 임계와 성능

- 방법: fold마다 `thr = threshold_for_segment_fpr(train 정상 점수, seg_uid, t_start, target, win_s=1.0)`(any 집계)로 임계를 정해 test에 적용한다. 목표 `(0.005, 0.01, 0.02, 0.04, 0.08)` × 모델 2 × split 2 = 20행. 이상 데이터는 임계 선택에 쓰지 않는다.
- `fp_per_hour = fpr_segment(test 실측) × SEG_PER_HOUR`, `felt_delay_median_s = delay_median_s + BURST_GAP_S`, `fpr_granularity = 1 / n_train_seg_mean`(train 정상 세그먼트 1개가 만드는 오경보율 단위).
- 참조: 31 `f1_sensitivity_grid.csv`의 q 방식(any, skip 0, q 0.99·0.995·0.999)과 나란히 놓는다. rule1 gkf 0.02·0.04 행은 31 `f1_matched_fpr.csv`와 tp/fp/fn/f1이 같아야 한다(assert).

In [4]:
TARGETS = (0.005, 0.01, 0.02, 0.04, 0.08)
rows = []
for m in MODELS:
    for sp in SPLITS:
        for tgt in TARGETS:
            c = run_global(m, sp, tgt)
            th = np.array(c["_thrs"], float)
            ntr = float(np.mean(c["_ntr"]))
            rows.append({"model": m, "split": sp, "target_fpr": tgt, "f1_agg": c["f1_agg"],
                         "thr_mean": float(th.mean()), "thr_min": float(th.min()), "thr_max": float(th.max()),
                         "thr_rel_range": float((th.max() - th.min()) / th.mean()),
                         **{key: c[key] for key in evaluate.F1_COLS},
                         "fpr_segment": c["fpr_segment"], "fp_per_hour": c["fpr_segment"] * SEG_PER_HOUR,
                         "delay_median_s": c["delay_median_s"], "felt_delay_median_s": c["delay_median_s"] + evaluate.BURST_GAP_S,
                         "n_train_seg_mean": ntr, "fpr_granularity": 1.0 / ntr})
CURVE = pd.DataFrame(rows)
assert len(CURVE) == 20
CURVE.to_csv(RES / "operating_curve.csv", index=False)

# assert: 목표가 커질수록 thr_mean 비증가·fp 비감소
for (m, sp), g in CURVE.groupby(["model", "split"]):
    g = g.sort_values("target_fpr")
    assert (np.diff(g.thr_mean.to_numpy()) <= 1e-12).all(), (m, sp, "thr_mean 증가")
    assert (np.diff(g.fp.to_numpy()) >= -1e-9).all(), (m, sp, "fp 감소")
print("단조성 assert 통과 (target 증가 -> thr_mean 비증가, fp 비감소)")

# assert: rule1 gkf 0.02·0.04 == 31 f1_matched_fpr (any, skip 0)
M31 = pd.read_csv(REF31 / "f1_matched_fpr.csv")
for tgt in (0.02, 0.04):
    a = CURVE[(CURVE.model == "rule1_rms_ai0") & (CURVE.split == "group_kfold_seg") & (CURVE.target_fpr == tgt)].iloc[0]
    b = M31[(M31.model == "rule1_rms_ai0") & (M31.split == "group_kfold_seg") & (M31.target_fpr == tgt) & (M31.rule == "any") & (M31.skip_first == 0)].iloc[0]
    print(f"rule1 gkf {tgt}: 32 tp/fp/fn/f1 = {a.tp:.0f}/{a.fp:.0f}/{a.fn:.0f}/{a.f1:.4f} | 31 = {b.tp:.0f}/{b.fp:.0f}/{b.fn:.0f}/{b.f1:.4f}")
    assert (a.tp, a.fp, a.fn) == (b.tp, b.fp, b.fn) and abs(a.f1 - b.f1) < 1e-12
print("rule1 gkf 0.02·0.04 == 31 f1_matched_fpr.csv 통과")
chk = []
for r in M31[(M31.rule == "any") & (M31.skip_first == 0)].itertuples():
    a = CURVE[(CURVE.model == r.model) & (CURVE.split == r.split) & (CURVE.target_fpr == r.target_fpr)].iloc[0]
    chk.append(abs(a.f1 - r.f1) < 1e-9 and abs(a.fp - r.fp) < 1e-9)
print("31 f1_matched_fpr.csv any·skip0 전체", len(chk), "행 중 f1·fp 일치:", int(sum(chk)))

SHOW = ["target_fpr", "thr_mean", "fpr_segment", "fp_per_hour", "recall", "f1", "fp", "fn", "delay_median_s", "felt_delay_median_s"]
for m in MODELS:
    for sp in SPLITS:
        print(f"\n== {m} | {sp} ==")
        print(CURVE[(CURVE.model == m) & (CURVE.split == sp)][SHOW].round(4).to_string(index=False))

# q 방식 참조
G31 = pd.read_csv(REF31 / "f1_sensitivity_grid.csv")
Q = G31[(G31.rule == "any") & (G31.skip_first == 0) & (G31.q.isin([0.99, 0.995, 0.999]))].copy()
Q["fp_per_hour"] = Q.fpr_segment * SEG_PER_HOUR
Q = Q[["model", "split", "q", "fpr_segment", "fp_per_hour", "recall", "f1", "fp", "fn"]].reset_index(drop=True)
print("\n[참조] q 분위수 방식 (31 f1_sensitivity_grid.csv, any·skip 0)")
print(Q.round(4).to_string(index=False))

단조성 assert 통과 (target 증가 -> thr_mean 비증가, fp 비감소)
rule1 gkf 0.02: 32 tp/fp/fn/f1 = 14/12/3/0.6512 | 31 = 14/12/3/0.6512
rule1 gkf 0.04: 32 tp/fp/fn/f1 = 14/21/3/0.5385 | 31 = 14/21/3/0.5385
rule1 gkf 0.02·0.04 == 31 f1_matched_fpr.csv 통과
31 f1_matched_fpr.csv any·skip0 전체 8 행 중 f1·fp 일치: 8

== rule1_rms_ai0 | group_kfold_seg ==
 target_fpr  thr_mean  fpr_segment  fp_per_hour  recall     f1   fp  fn  delay_median_s  felt_delay_median_s
      0.005    0.1500       0.0094       3.8996  0.8235 0.7778  5.0 3.0             0.9                8.858
      0.010    0.1452       0.0113       4.6795  0.8235 0.7568  6.0 3.0             0.9                8.858
      0.020    0.1365       0.0226       9.3589  0.8235 0.6512 12.0 3.0             0.9                8.858
      0.040    0.1274       0.0396      16.3781  0.8235 0.5385 21.0 3.0             0.9                8.858
      0.080    0.1196       0.0792      32.7563  0.8235 0.3836 42.0 3.0             0.9                8.858

== rule1_rms_ai

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
COL = {"rule1_rms_ai0": "#2a78d6", "pca_spe_amp18": "#eb6834"}
for ax, sp in zip(axes, SPLITS):
    ax2 = ax.twinx()
    for m in MODELS:
        g = CURVE[(CURVE.model == m) & (CURVE.split == sp)].sort_values("target_fpr")
        ax.plot(g.target_fpr, g.recall, "-o", color=COL[m], label=f"{m} recall")
        ax2.plot(g.target_fpr, g.fp_per_hour, ":s", color=COL[m], label=f"{m} 시간당 오경보")
    ax.set_xscale("log"); ax.set_xticks(TARGETS); ax.set_xticklabels([str(t) for t in TARGETS])
    ax.minorticks_off()
    ax.set_xlabel("목표 오경보율 (세그먼트, train 정상 기준)"); ax.set_ylabel("recall (실선)"); ax2.set_ylabel("시간당 오경보 수 (점선)")
    ax.set_ylim(0, 1.05); ax.grid(alpha=.3); ax.set_title(sp)
axes[0].legend(fontsize=7, loc="lower right")
fig.suptitle("운영 곡선 — 목표 오경보율별 recall과 시간당 오경보 (any 집계, 1초 윈도우)", fontsize=11)
paths.save_fig(FIG, "operating_curve.png")

saved figures\32_operating_point_CHS\operating_curve.png


### B. 운전 상태별 운영점

- 목표 `(0.01, 0.02, 0.04)`, 두 방식 비교. `global`은 A와 같다. `by_state`는 train 정상 윈도우를 `state_hat`별로 나눠 각 상태 윈도우만 포함한 세그먼트 집합에서 `threshold_for_segment_fpr`로 임계 2개를 정하고, test는 윈도우별 `score − thr[state_hat]`를 점수로 `thr=0`에서 `segment_alarms`(any)를 호출한다.
- 정상 FP 세그먼트의 상태는 세그먼트 윈도우 `state_hat` 최빈값(대표 상태)으로 센다. 상태별 분모는 해당 대표 상태의 test 정상 세그먼트 수이다(gkf 합산, time_block은 블록 평균).
- 해석 규칙: by_state가 고부하 오경보율을 낮추고 recall을 유지하는지만 본다. 결과 방향을 미리 정하지 않는다(31 C절에서 FP는 고부하 상단 꼬리에 집중).

In [6]:
B_TARGETS = (0.01, 0.02, 0.04)

def ff_state(tgt):
    def ff(trn, s_tr, te, s_te):
        sh, uid, ts = trn.state_hat.to_numpy(), trn.seg_uid.to_numpy(), trn.t_start.to_numpy()
        thr = np.array([evaluate.threshold_for_segment_fpr(s_tr[sh == st], uid[sh == st], ts[sh == st], tgt, win_s=1.0) for st in (0, 1)])
        assert not np.isnan(thr).any()
        adj = s_te - thr[te.state_hat.to_numpy()]
        an, ao = eval_fold({}, 0.0, te, adj)
        return an, ao, thr
    return ff

rows = []
for m in MODELS:
    for sp in SPLITS:
        for tgt in B_TARGETS:
            for method in ("global", "by_state"):
                c = run_global(m, sp, tgt) if method == "global" else combine(m, sp, ff_state(tgt))
                th = np.array(c["_thrs"], float)
                thh, thl = (th.mean(), th.mean()) if method == "global" else (th[:, 0].mean(), th[:, 1].mean())
                rows.append({"model": m, "split": sp, "target_fpr": tgt, "method": method,
                             "thr_high_mean": float(thh), "thr_low_mean": float(thl), "f1_agg": c["f1_agg"],
                             **{key: c[key] for key in evaluate.F1_COLS},
                             "fpr_segment": c["fpr_segment"], "fp_per_hour": c["fpr_segment"] * SEG_PER_HOUR,
                             "delay_median_s": c["delay_median_s"],
                             "fp_seg_high": c["fp_seg_high"], "fp_seg_low": c["fp_seg_low"],
                             "fpr_seg_high": c["fpr_seg_high"], "fpr_seg_low": c["fpr_seg_low"]})
STATE_OP = pd.DataFrame(rows)
assert len(STATE_OP) == 24
STATE_OP.to_csv(RES / "state_operating_point.csv", index=False)
for r in STATE_OP[STATE_OP.method == "global"].itertuples():       # global 행은 A와 같은 값
    a = CURVE[(CURVE.model == r.model) & (CURVE.split == r.split) & (CURVE.target_fpr == r.target_fpr)].iloc[0]
    assert abs(a.f1 - r.f1) < 1e-12 and abs(a.fp - r.fp) < 1e-12
SHOW = ["model", "split", "method", "thr_high_mean", "thr_low_mean", "fpr_segment", "fp_per_hour", "recall", "f1", "fp", "fn",
        "fp_seg_high", "fp_seg_low", "fpr_seg_high", "fpr_seg_low", "delay_median_s"]
for tgt in B_TARGETS:
    print(f"\n=== 목표 오경보율 {tgt}: global vs by_state ===")
    print(STATE_OP[STATE_OP.target_fpr == tgt][SHOW].round(4).to_string(index=False))


=== 목표 오경보율 0.01: global vs by_state ===
        model           split   method  thr_high_mean  thr_low_mean  fpr_segment  fp_per_hour  recall     f1   fp   fn  fp_seg_high  fp_seg_low  fpr_seg_high  fpr_seg_low  delay_median_s
rule1_rms_ai0 group_kfold_seg   global         0.1452        0.1452       0.0113       4.6795  0.8235 0.7568 6.00 3.00         4.00        2.00        0.0195       0.0062             0.9
rule1_rms_ai0 group_kfold_seg by_state         0.1516        0.1261       0.0132       5.4594  0.8235 0.7368 7.00 3.00         4.00        3.00        0.0195       0.0092             0.9
rule1_rms_ai0      time_block   global         0.1402        0.1402       0.0168       6.9281  0.8235 0.8567 1.75 3.00         1.50        0.25        0.0305       0.0044             0.9
rule1_rms_ai0      time_block by_state         0.1428        0.1320       0.0144       5.9713  0.8235 0.8626 1.50 3.00         1.25        0.25        0.0256       0.0044             0.9
pca_spe_amp18 group_kfo

In [7]:
t = STATE_OP[STATE_OP.target_fpr == 0.02]
fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=True)
for ax, (key, g) in zip(axes, t.groupby(["model", "split"], sort=False)):
    gg = g.set_index("method")
    x = np.arange(2); w = 0.36
    ax.bar(x - w / 2, [gg.loc["global", "fpr_seg_high"], gg.loc["global", "fpr_seg_low"]], w, label="global", color="#52514e")
    ax.bar(x + w / 2, [gg.loc["by_state", "fpr_seg_high"], gg.loc["by_state", "fpr_seg_low"]], w, label="by_state", color="#2a78d6")
    ax.axhline(0.02, color="#eb6834", ls="--", lw=1)
    ax.set_xticks(x); ax.set_xticklabels(["고부하", "저부하"]); ax.set_title(f"{key[0]}\n{key[1]}", fontsize=8); ax.grid(alpha=.3, axis="y")
axes[0].set_ylabel("세그먼트 오경보율 (test 실측)"); axes[0].legend(fontsize=8)
fig.suptitle("목표 0.02에서 운전 상태별 세그먼트 오경보율 — global vs by_state (점선 = 목표)", fontsize=11)
paths.save_fig(FIG, "state_fp_balance.png")

saved figures\32_operating_point_CHS\state_fp_balance.png


### C. 2단계 운영점 — 같은 점수에 주의·경보 임계 2개

- **사전 고정: 주의 = 목표 0.04, 경보 = 목표 0.01.** 근거: 31 A-2에서 q 0.99 ≈ 4%가 현재 규칙 수준이고, 1%는 FP 4~5개 수준이라 F1 0.85 이상이 기대된다는 산수(#34)에서 나온 값이며 결과를 보고 바꾸지 않는다.
- 검증: fold마다 `thr_alert ≥ thr_watch`(목표가 낮을수록 임계가 높다 — 단조성), 경보 세그먼트 ⊆ 주의 세그먼트.
- 세그먼트 표는 gkf 기준 판정 가능 이상 17세그먼트 × 모델.

In [8]:
WATCH_T, ALERT_T = 0.04, 0.01
GR = S.set_index("seg_uid")[["seg_id", "vib_grade", "cur_grade"]]

def _med(x):
    x = pd.Series(x).dropna()
    return float(x.median()) if len(x) else np.nan

def _merge_levels(aw, aa):
    mg = aw[["seg_uid", "alarm", "delay_in_window_s"]].merge(aa[["seg_uid", "alarm", "delay_in_window_s"]], on="seg_uid", suffixes=("_w", "_a"))
    assert len(mg) == len(aw) == len(aa)
    assert not (mg.alarm_a & ~mg.alarm_w).any(), "경보 세그먼트가 주의 세그먼트의 부분집합이 아님"
    return mg

def two_level_stats(aos_w, aos_a, sp):
    groups = [(pd.concat(aos_w, ignore_index=True), pd.concat(aos_a, ignore_index=True))] if sp == "group_kfold_seg" else list(zip(aos_w, aos_a))
    out = []
    for aw, aa in groups:
        mg = _merge_levels(aw[aw.n_windows > 0], aa[aa.n_windows > 0])
        esc = mg[mg.alarm_a]
        out.append({"watch_only": int((mg.alarm_w & ~mg.alarm_a).sum()), "alert": int(mg.alarm_a.sum()), "missed": int((~mg.alarm_w).sum()),
                    "dw": _med(mg.loc[mg.alarm_w, "delay_in_window_s_w"]), "da": _med(mg.loc[mg.alarm_a, "delay_in_window_s_a"]),
                    "gap": _med(esc.delay_in_window_s_a - esc.delay_in_window_s_w) if len(esc) else np.nan})
    return {k: (float(np.nanmean([o[k] for o in out])) if not all(np.isnan(o[k]) for o in out) else np.nan) for k in out[0]}

rows, SEGROWS = [], []
for m in MODELS:
    for sp in SPLITS:
        cw, ca = run_global(m, sp, WATCH_T), run_global(m, sp, ALERT_T)
        for i, (tw, ta) in enumerate(zip(cw["_thrs"], ca["_thrs"])):
            assert ta >= tw, (m, sp, i, tw, ta)
        agg = two_level_stats(cw["_aos"], ca["_aos"], sp)
        for nw, na in zip(cw["_ans"], ca["_ans"]):                      # 정상 쪽도 경보 ⊆ 주의
            x = nw.merge(na, on="seg_uid", suffixes=("_w", "_a"))
            assert not (x.alarm_a & ~x.alarm_w).any()
        rows.append({"model": m, "split": sp, "thr_watch_mean": cw["thr_mean"], "thr_alert_mean": ca["thr_mean"],
                     "fpr_watch": cw["fpr_segment"], "fpr_alert": ca["fpr_segment"],
                     "fp_per_hour_watch": cw["fpr_segment"] * SEG_PER_HOUR, "fp_per_hour_alert": ca["fpr_segment"] * SEG_PER_HOUR,
                     "recall_watch": cw["recall"], "recall_alert": ca["recall"], "f1_watch": cw["f1"], "f1_alert": ca["f1"],
                     "n_out_watch_only": agg["watch_only"], "n_out_alert": agg["alert"], "n_out_missed": agg["missed"],
                     "delay_watch_median_s": agg["dw"], "delay_alert_median_s": agg["da"], "escalation_gap_median_s": agg["gap"]})
        if sp == "group_kfold_seg":
            aw, aa = pd.concat(cw["_aos"], ignore_index=True), pd.concat(ca["_aos"], ignore_index=True)
            mg = _merge_levels(aw[aw.n_windows > 0], aa[aa.n_windows > 0])
            for r in mg.itertuples():
                SEGROWS.append({"seg_id": int(GR.loc[r.seg_uid, "seg_id"]), "model": m, "watch": bool(r.alarm_w), "alert": bool(r.alarm_a),
                                "delay_watch_s": r.delay_in_window_s_w, "delay_alert_s": r.delay_in_window_s_a,
                                "vib_grade": GR.loc[r.seg_uid, "vib_grade"], "cur_grade": GR.loc[r.seg_uid, "cur_grade"]})
TWO = pd.DataFrame(rows); assert len(TWO) == 4
TWO.to_csv(RES / "two_level_alarm.csv", index=False)
SEGT = pd.DataFrame(SEGROWS).sort_values(["model", "seg_id"]).reset_index(drop=True)
assert SEGT.groupby("model").size().eq(17).all()
SEGT.to_csv(RES / "two_level_segments.csv", index=False)
print("모든 fold에서 thr_alert >= thr_watch, 경보 세그먼트 ⊆ 주의 세그먼트 assert 통과")
print(f"주의 = 목표 {WATCH_T}, 경보 = 목표 {ALERT_T}")
print(TWO.round(4).T.to_string(header=False))
for m in MODELS:
    g = SEGT[SEGT.model == m]
    print(f"\n[{m}] gkf 이상 {len(g)}세그먼트: 경보 {int(g.alert.sum())} / 주의만 {int((g.watch & ~g.alert).sum())} / 미탐 {int((~g.watch).sum())}")
    print("  미탐 seg_id:", g[~g.watch].seg_id.tolist(), "| 주의만 seg_id:", g[g.watch & ~g.alert].seg_id.tolist())
print()
print(SEGT.round(2).to_string(index=False))

모든 fold에서 thr_alert >= thr_watch, 경보 세그먼트 ⊆ 주의 세그먼트 assert 통과
주의 = 목표 0.04, 경보 = 목표 0.01
model                      rule1_rms_ai0  rule1_rms_ai0    pca_spe_amp18  pca_spe_amp18
split                    group_kfold_seg     time_block  group_kfold_seg     time_block
thr_watch_mean                    0.1274         0.1277            4.451         4.8003
thr_alert_mean                    0.1452         0.1402           8.8068          8.281
fpr_watch                         0.0396         0.0408           0.0434         0.0426
fpr_alert                         0.0113         0.0168           0.0113         0.0118
fp_per_hour_watch                16.3781        16.8444           17.938        17.5966
fp_per_hour_alert                 4.6795         6.9281           4.6795         4.8712
recall_watch                      0.8235         0.8235              1.0            1.0
recall_alert                      0.8235         0.8235              1.0            1.0
f1_watch                       

In [9]:
g = SEGT[SEGT.model == "pca_spe_amp18"].sort_values("seg_id").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(7.5, 5.5))
for i, r in g.iterrows():
    if r.watch:
        ax.scatter(r.delay_watch_s, i, facecolors="none", edgecolors="#2a78d6", s=70, lw=1.6, zorder=3, label="주의 시각")
    if r.alert:
        ax.scatter(r.delay_alert_s, i, color="#d62a2a", s=40, zorder=4, label="경보 시각")
    if not r.watch:
        ax.scatter(0, i, marker="x", color="k", s=60, zorder=4, label="미탐")
h, l = ax.get_legend_handles_labels()
uniq = dict(zip(l, h)); ax.legend(uniq.values(), uniq.keys(), fontsize=8)
ax.set_yticks(range(len(g))); ax.set_yticklabels(g.seg_id.astype(int))
ax.set_xlabel("세그먼트 내 시각 (s, 윈도우 끝 기준)"); ax.set_ylabel("이상 세그먼트 seg_id"); ax.grid(alpha=.3)
ax.set_title("pca_spe_amp18 gkf — 이상 17세그먼트의 주의(○, 목표 0.04)·경보(●, 목표 0.01) 시각", fontsize=9)
paths.save_fig(FIG, "two_level_timeline.png")

saved figures\32_operating_point_CHS\two_level_timeline.png


### D. 임계 안정성과 입도

- fold별 `thr`(A의 20행 × fold)를 긴 표로 저장하고, 목표별 `thr_rel_range`, 실측 test fpr − target의 범위, `fpr_granularity`(train 정상 세그먼트 1개 = 오경보율 단위)를 본다.

In [10]:
rows = []
for m in MODELS:
    for sp in SPLITS:
        for tgt in TARGETS:
            c = run_global(m, sp, tgt)
            for (k, trn, te), thr, ntr in zip(split_iter(sp), c["_thrs"], c["_ntr"]):
                _, s_tr, _, _ = SCORES[(m, sp, k)]
                rows.append({"model": m, "split": sp, "target_fpr": tgt, "fold": k, "thr": float(thr), "n_train_seg": ntr,
                             "train_fpr_realized": train_fpr(trn, s_tr, thr)})
BYF = pd.DataFrame(rows); assert len(BYF) == 2 * (5 + 4) * len(TARGETS)
BYF.to_csv(RES / "threshold_by_fold.csv", index=False)
print("train 실측 오경보율 <= 목표 위반 행:", int((BYF.train_fpr_realized > BYF.target_fpr + 1e-12).sum()))
D = CURVE[["model", "split", "target_fpr", "thr_rel_range", "fpr_segment", "n_train_seg_mean", "fpr_granularity"]].copy()
D["test_fpr_minus_target"] = D.fpr_segment - D.target_fpr
tr = BYF.groupby(["model", "split", "target_fpr"]).train_fpr_realized.agg(train_fpr_min="min", train_fpr_max="max").reset_index()
D = D.merge(tr, on=["model", "split", "target_fpr"])
print(D.round(4).to_string(index=False))
print("\n[목표별 요약]")
print(D.groupby("target_fpr").agg(thr_rel_range_max=("thr_rel_range", "max"), thr_rel_range_mean=("thr_rel_range", "mean"),
                                  diff_min=("test_fpr_minus_target", "min"), diff_max=("test_fpr_minus_target", "max"),
                                  granularity_mean=("fpr_granularity", "mean")).round(4).to_string())
print("\ntrain 정상 세그먼트 수 범위:", int(BYF.n_train_seg.min()), "~", int(BYF.n_train_seg.max()))

train 실측 오경보율 <= 목표 위반 행: 0
        model           split  target_fpr  thr_rel_range  fpr_segment  n_train_seg_mean  fpr_granularity  test_fpr_minus_target  train_fpr_min  train_fpr_max
rule1_rms_ai0 group_kfold_seg       0.005         0.0516       0.0094             424.0           0.0024                 0.0044         0.0047         0.0048
rule1_rms_ai0 group_kfold_seg       0.010         0.0494       0.0113             424.0           0.0024                 0.0013         0.0093         0.0095
rule1_rms_ai0 group_kfold_seg       0.020         0.0484       0.0226             424.0           0.0024                 0.0026         0.0187         0.0190
rule1_rms_ai0 group_kfold_seg       0.040         0.0409       0.0396             424.0           0.0024                -0.0004         0.0380         0.0400
rule1_rms_ai0 group_kfold_seg       0.080         0.0236       0.0792             424.0           0.0024                -0.0008         0.0784         0.0800
rule1_rms_ai0      time_

## 결론

수치 출처: 위 셀 출력과 `results/32_operating_point_CHS/`의 CSV(`operating_curve.csv`·`state_operating_point.csv`·`two_level_alarm.csv`·`two_level_segments.csv`·`threshold_by_fold.csv`). 이상 이벤트 1건(판정 가능 17세그먼트)·모델 2개의 결과이며 일반화하지 않는다.

| 절 | 발견 | 시사점 | 활용 아이디어 |
|---|---|---|---|
| A 운영 곡선 | 정상 수집 구간 76.93분, 판정 가능 530세그먼트 → `SEG_PER_HOUR` 413.35(전체 599 기준 467.17). pca_spe_amp18 gkf는 목표 0.01에서 recall 1.0000·fp 6(실측 0.0113, 시간당 4.7회)·F1 0.8500, 0.04에서 fp 23(시간당 17.9회)·F1 0.5965. rule1 gkf는 목표를 올려도 recall 0.8235(미탐 3개)로 일정하고 fp만 5→42(F1 0.7778→0.3836). q 방식은 같은 q 0.99에서 실측 오경보율이 rule1 0.0396·pca 0.0585로 달랐다(`operating_curve.csv`: target_fpr·fpr_segment·fp_per_hour·recall·f1, 31 `f1_sensitivity_grid.csv`) | 목표 오경보율로 임계를 정하면 점수 종류와 무관하게 시간당 오경보 수가 같은 눈금에서 비교된다. 이 이벤트에서 목표를 높여도 탐지율은 늘지 않고(rule1 일정, pca는 0.01부터 1.0) 오경보만 는다 | 운영점은 "감당할 시간당 오경보 수"로 먼저 정하고 임계는 정상 점수에서 역산한다(목표 0.01 = 명목 약 4.1회/시간, 이 데이터 실측 4.7~6.9회/시간) |
| B 상태별 | 12개 비교(2모델×2 split×3 목표) 중 by_state가 고부하 오경보율을 낮춘 경우는 4개(모두 rule1), 같음 3개, 높임 5개이고 recall은 10개에서 유지(pca 목표 0.01 gkf 1.0000→0.9412, time_block 1.0000→0.9853에서 하락). 목표 0.02 rule1 gkf: 고부하 0.0488→0.0341·저부하 0.0062→0.0123·F1 0.6512→0.6667. pca gkf: 고부하 0.0146→0.0244·저부하 0.0308→0.0277·F1 0.7234→0.7083. by_state는 전체 실측 오경보율도 목표를 더 넘긴다(pca gkf 0.01: 0.0113→0.0170) (`state_operating_point.csv`: fpr_seg_high·fpr_seg_low·fpr_segment·recall·f1) | 상태별 임계는 일관된 이득이 없다. 오경보 집중 상태가 점수마다 다르다(rule1은 고부하, pca_spe_amp18은 저부하가 더 높음). 31 C절의 고부하 집중을 전 모델에 일반화할 수 없다 | 상태별 임계는 기본값으로 쓰지 않고, 점수별로 `fpr_seg_high`·`fpr_seg_low`를 확인한 뒤 한쪽 상태가 목표를 크게 넘을 때만 적용한다 |
| C 2단계 | 주의(목표 0.04)·경보(목표 0.01) 모두 모든 fold에서 thr_alert ≥ thr_watch, 경보 ⊆ 주의. 주의만 걸린 이상 세그먼트 0개(두 모델·두 split), 미탐은 rule1 gkf 3·19·20, pca_spe_amp18은 17개 전부 경보. pca gkf 오경보는 주의 fp 23(시간당 17.9회) vs 경보 fp 6(4.7회)이고 경보 지연은 seg 4 1.4초·8 1.9초·15 2.4초·20 1.4초로 주의(0.9초)보다 늦었다 (`two_level_alarm.csv`, `two_level_segments.csv`) | 이 이벤트에서는 주의 단계가 추가로 잡은 이상 세그먼트가 없고 오경보만 17세그먼트(gkf pca) 더 만들었다. 관찰된 주의의 이득은 pca gkf 4/17 세그먼트에서 0.5~1.5초 선행한 것뿐(체감 지연 하한 7.958초 대비 미미). 2단계의 가치는 탐지 추가가 아니라 대응 등급 분리(점검 목록 vs 즉시 조치)에 있으며, 이는 이 데이터로 검증하지 못했다 | 경보를 1차 조치 기준으로, 주의를 점검 우선순위 목록으로 운영하되 주의의 효과는 이상이 약하게 나타나는 사례가 쌓인 뒤 재평가한다 |
| D 안정성·입도 | fold별 임계 상대 범위 `thr_rel_range`는 rule1 0.0236~0.0791, pca 0.0989~0.5425(time_block 목표 0.005가 최대). train 정상 세그먼트 평균 424개(gkf)·266개(time_block)라 1개 = 0.24%p·0.38%p(`fpr_granularity`). test 실측 − 목표는 목표 0.01에서 +0.0013~+0.0068, 0.005에서 −0.0004~+0.0071, 0.08에서 −0.0135~+0.0196 (`operating_curve.csv`, `threshold_by_fold.csv`) | 목표 0.005는 train 정상 약 2개 수준(424×0.005≈2.1)이라 거칠고, pca 점수는 상위 꼬리 때문에 fold별 임계가 크게 흔들린다. 목표가 낮을수록 임계가 대체로 불안정하나 단조는 아니다(pca gkf 0.005 0.2984 < 0.01 0.3368); time_block 범위에는 train 113→428개 증가 효과가 섞여 있다 | 목표는 train 정상 세그먼트 1개 단위(gkf 0.24%p, 가장 작은 train 0.88%p)보다 큰 값으로 두되, train 정상이 적을 때(113개면 0.01 = 1개)는 0.02 이상이 안전하고 정상 데이터가 늘면 재산정한다 |

### 운영 절차 (심사 4번)

| 단계 | 조건 | 현장 조치 | 해제 | 근거 수준 |
|---|---|---|---|---|
| 주의 | 점수 > 주의 임계(목표 오경보 4%) | 점검 목록 등재, 다음 burst 확인 | 연속 N burst 정상 (N은 [가정], 예: 5) | [데이터] 오경보·탐지율(`two_level_alarm.csv`) / [가정] 조치 |
| 경보 | 점수 > 경보 임계(목표 1%) | 즉시 점검·정지 판단 | 담당자 확인 | 동일 |
| 재학습 기준 | 월별 정상 오경보율이 목표의 2배 초과 또는 운전 상태 분포 변화 | 정상 데이터 재수집 후 임계 재산정 | — | [가정] 배수 2는 D절 실측/목표 비(목표 0.01 최대 1.68, 0.005 2.43)를 넘는 값, 목표 0.01 이상에서만 의미 |

### 한계

- 이상 이벤트 1건(판정 가능 17세그먼트)이고 모델은 2개뿐이다. F1 신뢰구간은 만들지 않았다.
- 시간당 오경보는 정상 수집 구간 76.93분·판정 가능 530세그먼트(413.35/시간) 기준 외삽이며 실제 운영 시간대의 정상 변동은 반영하지 않는다.
- 운전 상태 추정은 윈도우 AC RMS 경계 하나(125.2054)이고 parquet 상태와의 일치율은 0.9442이다.
- 28의 개별 모델 임계에는 아직 적용하지 않았다(결합 후 임계 재설정은 28이 기각). 24~27 모델은 train 정상 점수가 없어 제외했다.

### 다음 단계

1. 24~27이 train 정상 점수를 포함한 윈도우 점수를 저장하면 같은 절차를 주력 모델로 확장한다.
2. 28의 개별 모델(CNN·MCD) 임계를 q99 대신 이 운영점으로 정한 뒤 AND 결합 성능을 본다.
3. 보고서 4번 절에 위 운영 절차 표를 반영한다.

In [11]:
RUNTIME_S = time.time() - T0
for f in ("operating_curve.csv", "state_operating_point.csv", "two_level_alarm.csv", "two_level_segments.csv", "threshold_by_fold.csv"):
    assert (RES / f).exists(), f
for f in ("operating_curve.png", "state_fp_balance.png", "two_level_timeline.png"):
    assert (FIG / f).exists(), f
print(f"CSV 5개·PNG 3개 존재 | 노트북 실행 시간 {RUNTIME_S:.1f}초")

CSV 5개·PNG 3개 존재 | 노트북 실행 시간 111.5초
